# AURORA CORE — Colab GPU Worker + Model Runtime

Connect a Google Colab GPU runtime to your AURORA CORE backend and run model inference.

**Steps:**
1. Runtime → Change runtime type → **GPU** (T4 recommended)
2. Run Cell 1 to configure
3. Run Cell 2 to detect GPU
4. Run Cell 3 to connect worker + start heartbeat + job polling
5. Run Cell 4 to run GPU benchmark
6. Run Cell 5 to load model (smollm2-1.7b)
7. Run Cell 6 to run inference
8. Run Cell 7 to check runtime status
9. Run Cell 8 to disconnect cleanly

**AURORA does NOT:**
- Access your Google account
- Automate login
- Store your credentials
- Accept arbitrary code execution

In [ ]:
#@title Cell 1: Configure Connection { display-mode: "form" }
#@markdown Enter your AURORA backend URL and worker token.
#@markdown The token is entered securely (not stored in notebook).

import getpass
import os

#@markdown ---
#@markdown **AURORA Backend URL** (production Render backend):
AURORA_BACKEND_URL = "https://aurora-core-1-txvl.onrender.com" #@param {type:"string"}
#@markdown ---

print(f"Backend URL: {AURORA_BACKEND_URL}")
print("\nEnter your AURORA worker token (input will be hidden):")
AURORA_WORKER_TOKEN = getpass.getpass("Worker token: ")

if not AURORA_WORKER_TOKEN:
    raise ValueError("Worker token cannot be empty")

print("\nConfiguration saved. Token is masked and not stored.")
print(f"Backend: {AURORA_BACKEND_URL}")
print(f"Token: {'*' * 8}{AURORA_WORKER_TOKEN[-4:] if len(AURORA_WORKER_TOKEN) > 4 else '****'}")

In [ ]:
#@title Cell 2: Detect GPU { display-mode: "form" }
#@markdown Detects GPU hardware from the Colab runtime.
#@markdown Only reports values actually detected by PyTorch CUDA or nvidia-smi.

import subprocess
import sys

GPU_INFO = {
    "name": "UNKNOWN",
    "vendor": "UNKNOWN",
    "vram_mb": 0.0,
    "cuda_version": None,
    "driver_version": None,
    "compute_capability": None,
    "available_memory_mb": 0.0,
    "runtime_info": None,
}

try:
    import torch
    if torch.cuda.is_available():
        GPU_INFO["name"] = torch.cuda.get_device_name(0)
        GPU_INFO["vendor"] = "NVIDIA"
        GPU_INFO["cuda_version"] = torch.version.cuda
        GPU_INFO["compute_capability"] = ".".join(str(x) for x in torch.cuda.get_device_capability(0))
        props = torch.cuda.get_device_properties(0)
        GPU_INFO["vram_mb"] = round(props.total_mem / (1024 * 1024), 1)
        GPU_INFO["available_memory_mb"] = round(
            (props.total_mem - torch.cuda.memory_allocated(0)) / (1024 * 1024), 1
        )
        GPU_INFO["runtime_info"] = f"PyTorch {torch.__version__}"
        print(f"GPU detected: {GPU_INFO['name']}")
        print(f"VRAM: {GPU_INFO['vram_mb']:.0f} MB ({GPU_INFO['vram_mb']/1024:.1f} GB)")
        print(f"CUDA: {GPU_INFO['cuda_version']}")
        print(f"Compute capability: {GPU_INFO['compute_capability']}")
        print(f"Runtime: {GPU_INFO['runtime_info']}")
    else:
        print("WARNING: torch.cuda.is_available() = False")
        print("No GPU detected. Go to Runtime → Change runtime type → GPU.")
except ImportError:
    print("PyTorch not available. Running nvidia-smi fallback...")
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5,
        )
        if result.returncode == 0:
            parts = result.stdout.strip().split(", ")
            if len(parts) >= 4:
                GPU_INFO["name"] = parts[0].strip()
                GPU_INFO["vendor"] = "NVIDIA"
                GPU_INFO["vram_mb"] = float(parts[1].strip())
                GPU_INFO["driver_version"] = parts[2].strip()
                GPU_INFO["compute_capability"] = parts[3].strip()
                GPU_INFO["available_memory_mb"] = GPU_INFO["vram_mb"]
                GPU_INFO["runtime_info"] = "nvidia-smi"
                print(f"GPU detected (nvidia-smi): {GPU_INFO['name']}")
                print(f"VRAM: {GPU_INFO['vram_mb']:.0f} MB")
            else:
                print("nvidia-smi returned unexpected format")
        else:
            print("nvidia-smi not available or failed")
    except Exception as e:
        print(f"GPU detection failed: {e}")

print(f"\nGPU status: {GPU_INFO['name']}")

In [ ]:
#@title Cell 3: Connect Worker + Start Heartbeat + Job Polling { display-mode: "form" }
#@markdown Connects to AURORA backend, registers worker, starts heartbeat and job polling.
#@markdown The job polling loop picks up runtime operations (load, infer, unload).

import hashlib
import json
import socket
import sys
import time
import threading
from typing import Any

import requests

PROTOCOL_VERSION = "2.0"
WORKER_VERSION = "0.2.0"
HEARTBEAT_INTERVAL = 30

WORKER_ID = f"colab-{socket.gethostname()}-{int(time.time())}"
START_TIME = time.time()
CONNECTED = False

SESSION = requests.Session()
SESSION.headers.update({"Content-Type": "application/json"})

def build_capabilities() -> dict:
    try:
        import torch
        pytorch_version = torch.__version__
    except ImportError:
        pytorch_version = None
    return {
        "inference": True,
        "embeddings": True,
        "vision": True,
        "training": True,
        "benchmark": True,
        "runtime": True,
        "max_concurrency": 1,
        "gpu": GPU_INFO,
        "framework": "pytorch",
        "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
        "pytorch_version": pytorch_version,
    }

def api_post(path: str, data: dict) -> dict | None:
    url = f"{AURORA_BACKEND_URL}{path}"
    try:
        resp = SESSION.post(url, json=data, timeout=15)
        if resp.status_code == 200:
            return resp.json()
        print(f"API error {path}: {resp.status_code} {resp.text[:200]}")
        return None
    except requests.RequestException as e:
        print(f"Request failed {path}: {e}")
        return None

def api_get(path: str) -> dict | None:
    url = f"{AURORA_BACKEND_URL}{path}"
    try:
        resp = SESSION.get(url, timeout=10)
        if resp.status_code == 200:
            return resp.json()
        return None
    except requests.RequestException:
        return None

# Register worker
registration = {
    "worker_id": WORKER_ID,
    "provider_id": "colab",
    "provider_type": "GOOGLE_COLAB",
    "capabilities": build_capabilities(),
    "protocol_version": PROTOCOL_VERSION,
    "worker_version": WORKER_VERSION,
    "started_at": START_TIME,
    "api_token": AURORA_WORKER_TOKEN,
}

print(f"Registering worker: {WORKER_ID}")
ack = api_post("/api/v1/compute/workers/register", registration)

if ack is None:
    print("FAILED: No response from backend")
elif not ack.get("accepted"):
    print(f"FAILED: Registration rejected — {ack.get('error')}")
else:
    CONNECTED = True
    print("\n" + "=" * 50)
    print("AURORA WORKER CONNECTED")
    print("=" * 50)
    print(f"Worker:      {WORKER_ID}")
    print(f"GPU:         {GPU_INFO['name']}")
    print(f"VRAM:        {GPU_INFO['vram_mb']:.0f} MB ({GPU_INFO['vram_mb']/1024:.1f} GB)")
    print(f"CUDA:        {GPU_INFO.get('cuda_version', 'UNKNOWN')}")
    print(f"Runtime:     {GPU_INFO.get('runtime_info', 'UNKNOWN')}")
    print("=" * 50)

# Heartbeat + job polling loop
HEARTBEAT_STOP = threading.Event()
HEARTBEAT_COUNT = 0

def heartbeat_loop():
    global HEARTBEAT_COUNT
    while not HEARTBEAT_STOP.is_set():
        heartbeat = {
            "worker_id": WORKER_ID,
            "status": "READY",
            "capabilities": build_capabilities(),
            "current_job_id": None,
            "uptime_seconds": time.time() - START_TIME,
            "api_token": AURORA_WORKER_TOKEN,
        }
        ack = api_post(f"/api/v1/compute/workers/{WORKER_ID}/heartbeat", heartbeat)
        if ack:
            HEARTBEAT_COUNT += 1
            if HEARTBEAT_COUNT % 5 == 0:
                print(f"Heartbeat #{HEARTBEAT_COUNT} sent")
        HEARTBEAT_STOP.wait(HEARTBEAT_INTERVAL)

heartbeat_thread = threading.Thread(target=heartbeat_loop, daemon=True)
heartbeat_thread.start()

print(f"\nHeartbeat loop started (interval: {HEARTBEAT_INTERVAL}s)")
print("Worker is now READY to accept runtime operations.")

In [ ]:
#@title Cell 4: Run GPU Benchmark { display-mode: "form" }
#@markdown Executes a real GPU matrix multiplication benchmark.

import hashlib

#@markdown ---
#@markdown Matrix size (N x N):
MATRIX_SIZE = 2048 #@param {type:"integer", min:256, max:8192}
#@markdown Number of iterations:
ITERATIONS = 20 #@param {type:"integer", min:1, max:100}
#@markdown ---

try:
    import torch
    if not torch.cuda.is_available():
        print("ERROR: No CUDA available.")
    else:
        print(f"Running GPU benchmark...")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        
        a = torch.randn(64, 64, device="cuda")
        b = torch.randn(64, 64, device="cuda")
        for _ in range(3):
            _ = torch.mm(a, b)
        torch.cuda.synchronize()

        start = time.time()
        a = torch.randn(MATRIX_SIZE, MATRIX_SIZE, device="cuda")
        b = torch.randn(MATRIX_SIZE, MATRIX_SIZE, device="cuda")
        for _ in range(ITERATIONS):
            _ = torch.mm(a, b)
        torch.cuda.synchronize()
        elapsed = time.time() - start

        total_flops = 2.0 * MATRIX_SIZE ** 3 * ITERATIONS
        gflops = total_flops / elapsed / 1e9
        checksum = hashlib.sha256(f"gpu-{MATRIX_SIZE}-{ITERATIONS}".encode()).hexdigest()[:16]

        print("=" * 50)
        print("GPU BENCHMARK RESULT")
        print("=" * 50)
        print(f"GPU:              {torch.cuda.get_device_name(0)}")
        print(f"Matrix:           {MATRIX_SIZE}x{MATRIX_SIZE}")
        print(f"Iterations:       {ITERATIONS}")
        print(f"Execution time:   {elapsed:.3f}s")
        print(f"GFLOPS:           {gflops:.2f}")
        print(f"Checksum:         {checksum}")
        print(f"Status:           PASSED")
        print("=" * 50)

        del a, b
        torch.cuda.empty_cache()

except Exception as e:
    print(f"Benchmark failed: {e}")

In [ ]:
#@title Cell 5: Load Model (smollm2-1.7b) { display-mode: "form" }
#@markdown Loads the approved model on the GPU. Takes 30-120 seconds.
#@markdown Requires: transformers, torch installed in Colab.

import time as _time

#@markdown ---
#@markdown Model to load:
MODEL_ID = "smollm2-1.7b" #@param ["smollm2-1.7b", "phi-3.5-mini"]
#@markdown ---

print(f"Loading model: {MODEL_ID}")
print(f"This will download ~{2 if MODEL_ID == 'smollm2-1.7b' else 4} GB to GPU...")
print("Starting load...\n")

LOAD_START = _time.time()

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    torch_dtype = torch.float16
    
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    print("Tokenizer loaded.")
    
    print("Loading model weights (this takes a while)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    
    LOAD_TIME = _time.time() - LOAD_START
    mem_used = torch.cuda.memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0
    
    print("\n" + "=" * 50)
    print("MODEL LOADED SUCCESSFULLY")
    print("=" * 50)
    print(f"Model:          {MODEL_ID}")
    print(f"Framework:      transformers + PyTorch")
    print(f"Device:         {model.device}")
    print(f"Dtype:          {torch_dtype}")
    print(f"Load time:      {LOAD_TIME:.1f}s")
    print(f"Memory used:    {mem_used:.0f} MB")
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:     {torch.cuda.get_device_properties(0).total_mem / (1024*1024):.0f} MB")
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / (1024*1024):.0f} MB")
    print(f"VRAM reserved:  {torch.cuda.memory_reserved() / (1024*1024):.0f} MB")
    print("=" * 50)
    
    MODEL_LOADED = True
    
except Exception as e:
    LOAD_TIME = _time.time() - LOAD_START
    print(f"\nModel load FAILED after {LOAD_TIME:.1f}s: {e}")
    MODEL_LOADED = False

In [ ]:
#@title Cell 6: Run Inference { display-mode: "form" }
#@markdown Runs a real inference on the loaded model via Tesla T4 GPU.

import hashlib
import time as _time

#@markdown ---
#@markdown Test prompt:
PROMPT = "Explain in two short sentences why evidence and uncertainty matter in scientific analysis." #@param {type:"string"}
#@markdown Max new tokens:
MAX_TOKENS = 128 #@param {type:"integer", min:16, max:1024}
#@markdown Temperature:
TEMPERATURE = 0.7 #@param {type:"number", min:0.0, max:2.0}
#@markdown ---

if not MODEL_LOADED:
    print("ERROR: No model loaded. Run Cell 5 first.")
else:
    prompt_hash = hashlib.sha256(PROMPT.encode()).hexdigest()[:16]
    print(f"Running inference on {torch.cuda.get_device_name(0)}...")
    print(f"Prompt: {PROMPT[:80]}{'...' if len(PROMPT) > 80 else ''}")
    print(f"Max tokens: {MAX_TOKENS}, Temperature: {TEMPERATURE}")
    print()

    INF_START = _time.time()
    
    try:
        inputs = tokenizer(PROMPT, return_tensors="pt")
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                top_p=0.9,
                do_sample=TEMPERATURE > 0,
            )
        
        generated = outputs[0][inputs["input_ids"].shape[-1]:]
        output_text = tokenizer.decode(generated, skip_special_tokens=True)
        INF_TIME = _time.time() - INF_START
        tokens = len(generated)
        output_hash = hashlib.sha256(output_text.encode()).hexdigest()[:16]
        
        print("=" * 60)
        print("LIVE MODEL INFERENCE RESULT")
        print("=" * 60)
        print(f"Status:           COMPLETED")
        print(f"Model:            {MODEL_ID}")
        print(f"Device:           CUDA (GPU)")
        print(f"GPU:              {torch.cuda.get_device_name(0)}")
        print(f"Duration:         {INF_TIME:.3f}s")
        print(f"Tokens generated: {tokens}")
        print(f"Tokens/sec:       {tokens / INF_TIME:.1f}")
        print(f"Input hash:       {prompt_hash}")
        print(f"Output hash:      {output_hash}")
        print(f"\n--- Generated Output ---")
        print(output_text)
        print("--- End Output ---")
        print("=" * 60)
        print("\nLIVE MODEL INFERENCE VERIFIED: YES")
        
    except Exception as e:
        INF_TIME = _time.time() - INF_START
        print(f"\nInference FAILED after {INF_TIME:.3f}s: {e}")
        print("LIVE MODEL INFERENCE VERIFIED: NO")

In [ ]:
#@title Cell 7: Check Runtime Status { display-mode: "form" }
#@markdown Checks runtime status via the AURORA API.

import json

print("Checking runtime health...")
health = api_get("/api/v1/runtime/health")
if health:
    print(f"\nRuntime Health:")
    print(f"  Status:       {health.get('status', 'UNKNOWN')}")
    print(f"  Runtimes:     {health.get('runtimes', 0)}")
    print(f"  Models:       {health.get('registry_models', 0)}")
    print(f"  Inferences:   {health.get('inference_jobs', 0)}")
else:
    print("Runtime API not available")

print("\nChecking runtimes...")
runtimes = api_get("/api/v1/runtime/runtimes")
if runtimes and runtimes.get("runtimes"):
    for rt in runtimes["runtimes"]:
        print(f"\n  Runtime: {rt.get('runtime_id', 'N/A')}")
        print(f"  Status:  {rt.get('status', 'N/A')}")
        print(f"  Worker:  {rt.get('worker_id', 'N/A')}")
        print(f"  GPU:     {rt.get('gpu_name', 'N/A')}")
        model = rt.get('model')
        if model:
            print(f"  Model:   {model.get('model_name', 'N/A')}")
            print(f"  Device:  {model.get('device', 'N/A')}")
else:
    print("  No active runtimes")

print("\nChecking compute status...")
compute = api_get("/api/v1/compute/status")
if compute:
    print(f"  Enabled:  {compute.get('enabled', False)}")
    print(f"  Mode:     {compute.get('mode', 'N/A')}")
    print(f"  GPU:      {'ON' if compute.get('gpu_enabled') else 'OFF'}")
    print(f"  Active:   {compute.get('active_provider', 'N/A')}")
    for p in compute.get("providers", []):
        if p.get("provider_type") == "GOOGLE_COLAB":
            print(f"  Colab:    {p.get('status', 'N/A')}")
            print(f"  Worker:   {p.get('worker_id', 'N/A')}")
            gpu = p.get("capabilities", {}).get("gpu")
            if gpu:
                print(f"  GPU name: {gpu.get('name', 'N/A')}")
                print(f"  VRAM:     {gpu.get('vram_mb', 0):.0f} MB")

In [ ]:
#@title Cell 8: Disconnect Worker { display-mode: "form" }
#@markdown Gracefully shuts down the worker and sends shutdown to AURORA.

print("Stopping heartbeat and job polling...")
HEARTBEAT_STOP.set()

print("Sending shutdown to backend...")
shutdown_data = {
    "worker_id": WORKER_ID,
    "reason": "User disconnected from Colab",
    "api_token": AURORA_WORKER_TOKEN,
}
result = api_post(f"/api/v1/compute/workers/{WORKER_ID}/shutdown", shutdown_data)

if result and result.get("shutdown"):
    print("Worker shut down successfully.")
else:
    print("Shutdown sent (backend may have already processed).")

CONNECTED = False
print(f"\nWorker {WORKER_ID} is now DISCONNECTED")
print("You can re-run Cell 3 to reconnect.")